In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Monitorare o annullare un job

Visualizza l'elenco dei tuoi workload nella [pagina Workloads](https://quantum.cloud.ibm.com/workloads).

## Visualizzare lo stato di un job
Vai alla [tabella Workloads](https://quantum.cloud.ibm.com/workloads) e controlla nella colonna Status se un job è stato completato o ha restituito un errore.

## Visualizzare l'utilizzo residuo
Vai alla [tabella Instances](https://quantum.cloud.ibm.com/instances) e seleziona la scheda associata al piano di cui vuoi visualizzare l'utilizzo residuo. Vengono mostrati il tempo totale utilizzato e il tempo totale rimanente nel tuo piano.

## Visualizzare le metriche sul numero di job e workload inviati
Vai alla [pagina Analytics](https://quantum.cloud.ibm.com/analytics) per vedere il numero totale di job inviati, nonché il conteggio dei workload batch e dei workload di sessione. Nota che puoi visualizzare la pagina Analytics solo per gli account che possiedi o gestisci.

## Monitorare un job
Usa l'istanza del job per controllarne lo stato o recuperarne i risultati chiamando il comando appropriato:

|                               |                                                                                                                                                                                                                                    |
| ----------------------------- | ---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| job.result()                  | Consulta i risultati del job immediatamente dopo il suo completamento. I risultati sono disponibili solo una volta che il job è terminato. Pertanto, `job.result()` è una chiamata bloccante fino al completamento del job.        |
| job.job\_id()                 | Restituisce l'ID che identifica univocamente il job. Per recuperare i risultati in un secondo momento è necessario l'ID del job. Si consiglia quindi di salvare gli ID dei job che potresti voler recuperare in seguito.            |
| job.status()                  | Controlla lo stato del job.                                                                                                                                                                                                        |
| job = service.job(\<job\_id>) | Recupera un job inviato in precedenza. Questa chiamata richiede l'ID del job.                                                                                                                                                      |

<span id="retrieve-later"></span>
## Recuperare i risultati di un job in un secondo momento
Chiama `service.job(\<job\_id>)` per recuperare un job inviato in precedenza. Se non hai l'ID del job, o se vuoi recuperare più job contemporaneamente — inclusi job eseguiti su QPU (unità di elaborazione quantistica) in pensione — chiama invece `service.jobs()` con filtri opzionali. Consulta [QiskitRuntimeService.jobs](../api/qiskit-ibm-runtime/qiskit-runtime-service#jobs).

> **Note:** `service.jobs()` restituisce anche i job eseguiti con il pacchetto deprecato `qiskit-ibm-provider`. I job inviati con il pacchetto ancora più datato (anch'esso deprecato) `qiskit-ibmq-provider` non sono più disponibili.

### Esempio
Questo esempio restituisce i 10 job runtime più recenti eseguiti su `my_backend`:

In [1]:
# This cell is hidden from users
from qiskit import QuantumCircuit
from qiskit.circuit import Parameter
from qiskit.transpiler import generate_preset_pass_manager

from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2
import numpy as np


my_backend = "ibm_torino"
service = QiskitRuntimeService()
# backend = service.backend(my_backend)
backend = service.least_busy()

# Define two circuits, each with one parameter with two parameters.
circuit = QuantumCircuit(2)
circuit.h(0)
circuit.cx(0, 1)
circuit.ry(Parameter("a"), 0)
circuit.cx(0, 1)
circuit.h(0)
circuit.measure_all()


pm = generate_preset_pass_manager(optimization_level=1, backend=backend)
transpiled_circuit = pm.run(circuit)

params = np.random.uniform(size=(2, 3)).T

sampler_pub = (transpiled_circuit, params)

# Instantiate the new estimator object, then run the transpiled circuit
# using the set of parameters and observables.
sampler = SamplerV2(mode=backend)
job = sampler.run([sampler_pub], shots=4)
print(job.job_id())

d305ck0ocacs73ajagvg


In [2]:
result = job.result()


spans = job.result().metadata["execution"]["execution_spans"]
print(spans)

ExecutionSpans([DoubleSliceSpan(<start='2025-09-09 16:31:16', stop='2025-09-09 16:31:16', size=24>)])


In [3]:
params = np.random.uniform(size=(2, 3))
params

array([[0.2260416 , 0.8747859 , 0.44361995],
       [0.94700856, 0.96826017, 0.98426562]])

In [4]:
mask = spans[0].mask(0)
mask

array([[[ True,  True,  True,  True],
        [ True,  True,  True,  True]],

       [[ True,  True,  True,  True],
        [ True,  True,  True,  True]],

       [[ True,  True,  True,  True],
        [ True,  True,  True,  True]]])

In [ ]:
from qiskit_ibm_runtime import QiskitRuntimeService

# Initialize the account first.
service = QiskitRuntimeService()
# Use `limit` to retrieve a specific number of jobs. The default `limit` is 10.
service.jobs(backend_name=my_backend)

## Cancel a job

You can cancel a job from the IBM Quantum Platform dashboard either on the Workloads page or the details page for a specific workload. On the Workloads page, click the overflow menu at the end of the row for that workload, and select Cancel. If you are on the details page for a specific workload, use the Actions dropdown at the top of the page, and select Cancel.

In Qiskit, use `job.cancel()` to cancel a job.

<span id="execution-spans"></span>
## View Sampler execution spans

The results of [`SamplerV2`](/docs/api/qiskit-ibm-runtime/sampler-v2) jobs executed in Qiskit Runtime contain execution timing information in their metadata.
This timing information can be used to place upper and lower timestamp bounds on when particular shots were executed on the QPU.
Shots are grouped into [`ExecutionSpan`](/docs/api/qiskit-ibm-runtime/execution-span-execution-span) objects, each of which indicates a start time, a stop time, and a specification of which shots were collected in the span.

An execution span specifies which data was executed during its window by providing an [`ExecutionSpan.mask`](/docs/api/qiskit-ibm-runtime/execution-span-execution-span#mask) method. This method, given any [Primitive Unified Block (PUB)](/docs/guides/primitive-input-output) index, returns a boolean mask that is `True` for all shots executed during its window. PUBs are indexed by the order in which they were given to the Sampler run call. If, for example, a PUB has shape `(2, 3)` and was run with four shots, then the mask's shape is `(2, 3, 4)`. See the [execution_span](/docs/api/qiskit-ibm-runtime/execution-span) API page for full details.

Example:

To view execution span information, review the metadata of the result returned by `SamplerV2`, which comes in the form of an `ExecutionSpans` object. This object is a list-like container containing instances of subclasses of `ExecutionSpan`, such as `SliceSpan`:

In [6]:
from qiskit.primitives import BitArray

# Get the mask of the 1st PUB for the 0th span.
mask = spans[0].mask(0)

# Decide whether the 0th shot of parameter set (1, 2) occurred in this span.
in_this_span = mask[2, 1, 0]

# Create a new bit array containing only the PUB-1 data collected during this span.
bits = result[0].data.meas
filtered_data = BitArray(bits.array[mask], bits.num_bits)

Execution spans can be filtered to include information pertaining to specific PUBs, selected by their indices:

In [7]:
# take the subset of spans that reference data in PUBs 0 or 2
spans.filter_by_pub([0, 2])

ExecutionSpans([DoubleSliceSpan(<start='2025-09-09 16:31:16', stop='2025-09-09 16:31:16', size=24>)])

View global information about the collection of execution spans:

In [8]:
print("Number of execution spans:", len(spans))
print("  Start of the first span:", spans.start)
print("     End of the last span:", spans.stop)
print("       Total duration (s):", spans.duration)

Number of execution spans: 1
  Start of the first span: 2025-09-09 16:31:16.320568
     End of the last span: 2025-09-09 16:31:16.865858
       Total duration (s): 0.54529


Extract and inspect a particular span:

In [9]:
spans.sort()
print(" Start of first span:", spans[0].start)
print("   End of first span:", spans[0].stop)
print("#shots in first span:", spans[0].size)

 Start of first span: 2025-09-09 16:31:16.320568
   End of first span: 2025-09-09 16:31:16.865858
#shots in first span: 24


## Annullare un job
Puoi annullare un job dalla dashboard di IBM Quantum Platform, sia dalla pagina Workloads sia dalla pagina di dettaglio di un workload specifico. Nella pagina Workloads, fai clic sul menu overflow alla fine della riga del workload e seleziona Cancel. Se ti trovi nella pagina di dettaglio di un workload specifico, usa il menu a tendina Actions in cima alla pagina e seleziona Cancel.

In Qiskit, usa `job.cancel()` per annullare un job.

<span id="execution-spans"></span>
## Visualizzare gli execution span del Sampler
I risultati dei job [`SamplerV2`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/sampler-v2) eseguiti in Qiskit Runtime contengono informazioni sui tempi di esecuzione nei loro metadati.
Queste informazioni temporali possono essere utilizzate per definire i limiti superiori e inferiori del timestamp relativo a quando determinate shot sono state eseguite sulla QPU.
Le shot sono raggruppate in oggetti [`ExecutionSpan`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/execution-span-execution-span), ognuno dei quali indica un orario di inizio, un orario di fine e una specifica di quali shot sono state raccolte nello span.

Uno execution span specifica quali dati sono stati eseguiti durante la sua finestra temporale tramite il metodo [`ExecutionSpan.mask`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/execution-span-execution-span#mask). Questo metodo, dato un qualsiasi indice di [Primitive Unified Block (PUB)](/guides/primitive-input-output), restituisce una maschera booleana che vale `True` per tutte le shot eseguite durante la finestra. I PUB sono indicizzati nell'ordine in cui sono stati passati alla chiamata run del Sampler. Se, ad esempio, un PUB ha forma `(2, 3)` ed è stato eseguito con quattro shot, la forma della maschera sarà `(2, 3, 4)`. Consulta la pagina API di [execution_span](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/execution-span) per tutti i dettagli.

Esempio:
Per visualizzare le informazioni sugli execution span, esamina i metadati del risultato restituito da `SamplerV2`, che si presentano sotto forma di oggetto `ExecutionSpans`. Questo oggetto è un contenitore simile a una lista che contiene istanze di sottoclassi di `ExecutionSpan`, come `SliceSpan`: